In [273]:
#Load Data from CSV
from pathlib import Path

LPS_data_folder = Path('LPS Prelim Data')
# Filter using .is_file() to exclude subfolders
LPS_data_filenames = [str(f) for f in LPS_data_folder.glob("*.csv") if f.is_file()]
print(LPS_data_filenames)

PIC_data_folder = Path('pIC Prelim Data')
PIC_data_filenames = [str(f) for f in PIC_data_folder.glob("*.csv") if f.is_file()]
print(PIC_data_filenames)


['LPS Prelim Data/CD11B.csv', 'LPS Prelim Data/CC3_Ly6G.csv', 'LPS Prelim Data/S100A9_CD3.csv']
['pIC Prelim Data/CD3_EGR1.csv', 'pIC Prelim Data/CD45_S100A9.csv']


In [277]:
#Sort Data into Tables 
import pandas as pd
import re

def format_df(df, markers):
    print(markers)
    #print(df.head())

    marker1 = markers[0]

    marker1_df = df[["Time", "Condition", "Injection", "Section", marker1 + ' Density', marker1 + ' LP Density']].copy()
    marker1_df.rename(columns={marker1 + ' Density': 'OE'}, inplace=True)
    marker1_df.rename(columns={marker1 + ' LP Density': 'LP'}, inplace=True)
    marker1_df.dropna(inplace=True)
    
    if len(markers) > 1:
        marker2 = markers[1]
        marker2_df = df[["Time", "Condition", "Injection", "Section", marker2 + ' Density', marker2 + ' LP Density']].copy()
        marker2_df.rename(columns={marker2 + ' Density': 'OE'}, inplace=True)
        marker2_df.rename(columns={marker2 + ' LP Density': 'LP'}, inplace=True)
        marker2_df.dropna(inplace=True)
    else:
        marker2_df = None

    return marker1_df, marker2_df

def make_dfs_from_files(filenames):
    marker_dfs = {}
    for file_path in filenames:
        df = pd.read_csv(file_path)
        filename = file_path.split("/")[1].split(".")[0]
        markers = filename.split("_")
        marker1_df, marker2_df = format_df(df, markers)

        marker_dfs[markers[0]] = marker1_df
        if len(markers) > 1:
            marker_dfs[markers[1]] = marker2_df
    return marker_dfs

LPS_marker_dfs = make_dfs_from_files(LPS_data_filenames)
print(LPS_marker_dfs)

PIC_marker_dfs = make_dfs_from_files(PIC_data_filenames)
print(PIC_marker_dfs)

['CD11B']
['CC3', 'Ly6G']
['S100A9', 'CD3']
{'CD11B':    Time Condition Injection Section           OE           LP
0    2d       LPS         C       B   209.378611    46.528580
1    2d       LPS         C       N     0.000000     0.000000
2    2d       LPS         C       T     0.000000     0.000000
3    2d       LPS         C       M     0.000000    52.431053
4    2d       LPS         C       X     0.000000   166.965180
5    2d       LPS         T       M     0.000000   136.892539
6    2d       LPS         T       X   173.818577   543.183053
7    2d       LPS         T       B   726.342289   396.186703
8    2d       LPS         T       N     0.000000     0.000000
9    2d       LPS         T       T   275.479986   206.609989
10   5d       LPS         C       B    89.044312   607.120307
11   5d       LPS         C       N     0.000000     0.000000
12   5d       LPS         C       T   187.459918   443.984017
13   5d       LPS         C       M     0.000000   106.710299
14   5d       LP

Sorting based on:
- Side (C, T)
- Section/NQO1 Identity (D, V)
- Epithelium invasion (OE, LP)
- Region (T, B, X, M, N) or (DR, DM, VM, VLA, VLB)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['font.family'] = 'DejaVu Sans'

do_save = True
set_y_range = 0

def generate_plot(
    groups: dict,
    title: str = "Value",
    ylabel: str = "Value",
    figsize: tuple = None,
    x_spacing: float = 1.0,
    jitter_width: float = 0.1,
    marker_size: int = 30,
    broken: bool = True,
    gap_threshold: float = 0.3,   # fraction of total range to trigger a break
    break_padding: float = 1.5,   # padding around each cluster (in data units)
    point_colors: dict = None,
    bool_offset: float = 0.2,
    color_false: str = 'white',
    color_true: str = 'black',
    color_legend: tuple = None,
    save_path: str = None,
):
    """
    Broken-axis scatter plot for any number of groups.

    Parameters
    ----------
    groups : dict
        Mapping of group label -> list/array of values.
        e.g. {"Mother": [13,14,18], "Father": [30,30,12]}
    ylabel : str
        Y-axis label.
    figsize : tuple, optional
        Figure size. Defaults to (2 + 0.6*n_groups, 3.5).
    jitter_width : float
        Half-width of horizontal jitter applied to each point.
    marker_size : int
        Scatter marker size (s= argument).
    gap_threshold : float
        Fraction of the total data range that, when exceeded by a gap
        between value clusters, triggers a break. Lower = more breaks.
    break_padding : float
        Extra space (data units) added above/below each cluster window.
    save_path : str, optional
        If given, saves the figure to this path instead of showing it.
    """
    labels = list(groups.keys())
    all_values = np.concatenate([np.asarray(v) for v in groups.values()])
    n_groups = len(labels)

    # ------------------------------------------------------------------
    # Per-point Fill Colours
    # ------------------------------------------------------------------
    fill_colors = {}
    bool_flags = {}
    for label, vals in groups.items():
        if point_colors and label in point_colors:
            flags = np.asarray(point_colors[label], dtype=bool)
        else:
            flags = np.zeros(len(vals), dtype=bool)
        fill_colors[label] = [color_true if f else color_false for f in flags]
        bool_flags[label] = flags


    # ------------------------------------------------------------------
    # Detect clusters of values separated by large gaps
    # ------------------------------------------------------------------

    if not broken:
        data_range = all_values.max() - all_values.min() or 1
        pad = data_range * 0.05
        clusters = [(all_values.min() - pad, all_values.max() + pad)]
    else:
        sorted_vals = np.sort(np.unique(all_values))
        total_range = sorted_vals[-1] - sorted_vals[0] if len(sorted_vals) > 1 else 1
        min_gap = gap_threshold * total_range

        # Build clusters: list of (low, high) pairs in data space
        clusters = []
        cluster_start = sorted_vals[0]
        prev = sorted_vals[0]
        for v in sorted_vals[1:]:
            if v - prev > min_gap:
                clusters.append((cluster_start - break_padding,
                                prev + break_padding))
                cluster_start = v
            prev = v
        pad = max(break_padding, (prev-cluster_start) * 0.1)
        clusters.append((cluster_start - pad, prev + pad))

    n_panels = len(clusters)

    # ------------------------------------------------------------------
    # Figure layout — panels sized proportionally to their data range
    # ------------------------------------------------------------------
    if figsize is None:
        figsize = (2 + 0.6 * n_groups, 3.5)

    fig = plt.figure(figsize=figsize)

    left   = 0.22
    right  = 0.90
    bottom = 0.14
    top    = 0.94
    gap    = 0.06          # vertical gap between panels

    total_height = top - bottom - gap * (n_panels - 1)
    spans = [c[1] - c[0] for c in clusters]
    total_span = sum(spans)
    heights = [s / total_span * total_height for s in spans]
    
    axes = []
    y_cursor = bottom
    for i, (cluster, h) in enumerate(zip(clusters, heights)):
        ax = fig.add_axes([left, y_cursor, right - left, h])
        if set_y_range == 0:   
            ax.set_ylim(*cluster)
        else:
            ax.set_ylim(-5, set_y_range)
        ax.yaxis.set_major_locator(plt.MaxNLocator(nbins=5, steps=[1,2,5,10]))
        axes.append(ax)
        y_cursor += h + gap

    axes = axes[::-1]   # top panel first, bottom panel last
    clusters = clusters[::-1]

    # ------------------------------------------------------------------
    # Plot data on every panel (only visible points will show)
    # ------------------------------------------------------------------
    rng = np.random.default_rng(0)  # reproducible jitter

    for ax, (ylo, yhi) in zip(axes, clusters):
        for xi, (label, vals) in enumerate(groups.items()):
            vals = np.asarray(vals)
            colors = fill_colors[label]
            jitter = rng.uniform(-jitter_width, jitter_width, len(vals))
            for j, (v, c, flag) in enumerate(zip(vals, colors, flags)):
                x_offset = bool_offset if flag else -bool_offset
                ax.scatter(
                    xi * x_spacing + x_offset + jitter[j], v,
                    color=c, edgecolors='black',
                    linewidths=0.8, s=marker_size, zorder=3,
                )

        ax.set_xlim(-0.5, n_groups - 0.5)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    # ------------------------------------------------------------------
    # Spine / tick cosmetics
    # ------------------------------------------------------------------
    for i, ax in enumerate(axes):
        is_bottom = (i == len(axes) - 1)
        is_top    = (i == 0)
        ax.tick_params(axis='y', labelsize=10)

        if not is_top:
            ax.spines['top'].set_visible(False)
        if not is_bottom:
            ax.spines['bottom'].set_visible(False)
            ax.tick_params(bottom=False, labelbottom=False)

        if is_bottom:
            ax.set_xticks(range(n_groups))
            ax.set_xticklabels(labels, fontsize=10)
        else:
            ax.tick_params(bottom=False, labelbottom=False)

    # Y-axis label on the middle panel
    mid_ax = axes[len(axes) // 2]
    mid_ax.set_ylabel(ylabel, fontsize=12)

    # ------------------------------------------------------------------
    # Break marks (diagonal slash lines between panels)
    # ------------------------------------------------------------------
    d       = 0.013
    gap_y   = 0.004

    def _draw_break(ax, y_pos):
        trans = ax.transAxes
        for dy in [-gap_y, +gap_y]:
            for color, lw, zorder in [('white', 2.5, 10), ('black', 1.0, 11)]:
                line = plt.Line2D(
                    [-d, d], [y_pos + dy, y_pos + dy],
                    transform=trans, color=color, linewidth=lw,
                    clip_on=False, zorder=zorder,
                )
                ax.add_line(line)

    for i in range(n_panels - 1):
        _draw_break(axes[i],     0)   # bottom edge of upper panel
        _draw_break(axes[i + 1], 1)   # top edge of lower panel

    plt.title(label=title, fontsize=15)

    if color_legend:
        false_label, true_label = color_legend
        legend_handles = [
            plt.scatter([], [], color=color_false, edgecolors='black',
                        linewidths=0.8, s=marker_size, label=false_label),
            plt.scatter([], [], color=color_true, edgecolors='black',
                        linewidths=0.8, s=marker_size, label=true_label),
        ]
        axes[0].legend(
            handles=legend_handles,
            frameon=False,
            fontsize=10,
            loc='upper right',
            bbox_to_anchor=(1,1.35),
        )

    # ------------------------------------------------------------------
    # Save or show
    # ------------------------------------------------------------------
    if save_path and do_save == True:
        plt.savefig(save_path + ".pdf", bbox_inches="tight")
        print(f"Saved to {save_path}")
    else:
        plt.show()

    return fig, axes

In [324]:
#Generate Data Summaries
from functools import reduce

def intersect_dfs(dfs):
    return reduce(lambda left, right: pd.merge(left, right, how='inner'), dfs)

#Cell Density in OE v LP over all time points
def plot_oe_v_lp(df, marker, condition, labels):
    #print(df.head())

    control_df = df[df['Injection'] == "C"]
    treatment_df = df[df['Injection'] == "T"]

    if condition == 'LPS':
        #Section Names using LPS convention (TBXMN)
        dorsal_df = df[(df['Section'] == "N") | (df['Section'] == "M")]
        ventral_df = df[(df['Section'] == "B") | (df['Section'] == "T") | (df['Section'] == "X")]

        timepoint1 = df[(df['Time'] == "6h") & (df['Condition'] == "Saline")]
        timepoint2 = df[(df['Time'] == "6h") & (df['Condition'] == condition)]
        timepoint3 = df[(df['Time'] == "2d") & (df['Condition'] == condition)]
        timepoint4 = df[(df['Time'] == "5d") & (df['Condition'] == condition)]

    elif condition == 'PolyIC':
        #Section Names using Proper Convention (DR, DM, VM, VLA, VLB)
        dorsal_df = df[(df['Section'] == "DR") | (df['Section'] == "DM")]
        ventral_df = df[(df['Section'] == "VLA") | (df['Section'] == "VLB") | (df['Section'] == "VM")]

        timepoint1 = df[(df['Time'] == "6h") & (df['Condition'] == condition)]
        timepoint2 = df[(df['Time'] == "2d") & (df['Condition'] == condition)]
        timepoint3 = df[(df['Time'] == "5d") & (df['Condition'] == condition)]
        timepoint4 = df[(df['Time'] == "8d") & (df['Condition'] == condition)]

    series1_t_v = intersect_dfs([treatment_df, ventral_df, timepoint1])
    series2_t_v = intersect_dfs([treatment_df, ventral_df, timepoint2])
    series3_t_v = intersect_dfs([treatment_df, ventral_df, timepoint3])
    series4_t_v = intersect_dfs([treatment_df, ventral_df, timepoint4])

    generate_plot(
        groups={
            labels[0]: series1_t_v['OE'].tolist() + series1_t_v['LP'].tolist(),
            labels[1]: series2_t_v['OE'].tolist() + series2_t_v['LP'].tolist(),
            labels[2]: series3_t_v['OE'].tolist() + series3_t_v['LP'].tolist(),
            labels[3]: series4_t_v['OE'].tolist() + series4_t_v['LP'].tolist(),
        },
        title= condition + " " + marker + " OE vs LP Cell Density",
        ylabel="Cells / mm^2",
        broken=False,
        bool_offset=0.15,
        point_colors={
            labels[0]: [False] * len(series1_t_v['OE'].tolist()) + [True] * len(series1_t_v['LP'].tolist()),
            labels[1]: [False] * len(series2_t_v['OE'].tolist()) + [True] * len(series2_t_v['LP'].tolist()),
            labels[2]: [False] * len(series3_t_v['OE'].tolist()) + [True] * len(series3_t_v['LP'].tolist()),
            labels[3]: [False] * len(series4_t_v['OE'].tolist()) + [True] * len(series4_t_v['LP'].tolist())
        },
        color_false='orange',
        color_true='blue',
        color_legend=("Olfactory Epithelium", "Lamina Propria"),
        figsize=(8, 3.5),
        save_path="Figures/" + condition + "_OEvsLP_" + marker
    )

#Plot Cell Density in C vs T side for each timepoint
def plot_c_vs_t(df, marker, condition, labels, area):
    area = "OE" if area is None else area

    control_df = df[df['Injection'] == "C"]
    treatment_df = df[df['Injection'] == "T"]

    if condition == 'LPS':
        #Section Names using LPS convention (TBXMN)
        dorsal_df = df[(df['Section'] == "N") | (df['Section'] == "M")]
        ventral_df = df[(df['Section'] == "B") | (df['Section'] == "T") | (df['Section'] == "X")]

        timepoint1 = df[(df['Time'] == "6h") & (df['Condition'] == "Saline")]
        timepoint2 = df[(df['Time'] == "6h") & (df['Condition'] == condition)]
        timepoint3 = df[(df['Time'] == "2d") & (df['Condition'] == condition)]
        timepoint4 = df[(df['Time'] == "5d") & (df['Condition'] == condition)]

    elif condition == 'PolyIC':
        #Section Names using Proper Convention (DR, DM, VM, VLA, VLB)
        dorsal_df = df[(df['Section'] == "DR") | (df['Section'] == "DM")]
        ventral_df = df[(df['Section'] == "VLA") | (df['Section'] == "VLB") | (df['Section'] == "VM")]

        timepoint1 = df[(df['Time'] == "6h") & (df['Condition'] == condition)]
        timepoint2 = df[(df['Time'] == "2d") & (df['Condition'] == condition)]
        timepoint3 = df[(df['Time'] == "5d") & (df['Condition'] == condition)]
        timepoint4 = df[(df['Time'] == "8d") & (df['Condition'] == condition)]

    series1_c_v = intersect_dfs([control_df, ventral_df, timepoint1])[area]
    series2_c_v = intersect_dfs([control_df, ventral_df, timepoint2])[area]
    series3_c_v = intersect_dfs([control_df, ventral_df, timepoint3])[area]
    series4_c_v = intersect_dfs([control_df, ventral_df, timepoint4])[area]

    series1_t_v = intersect_dfs([treatment_df, ventral_df, timepoint1])[area]
    series2_t_v = intersect_dfs([treatment_df, ventral_df, timepoint2])[area]
    series3_t_v = intersect_dfs([treatment_df, ventral_df, timepoint3])[area]
    series4_t_v = intersect_dfs([treatment_df, ventral_df, timepoint4])[area]

    generate_plot(
        groups={
            labels[0]: series1_c_v.tolist() + series1_t_v.tolist(),
            labels[1]: series2_c_v.tolist() + series2_t_v.tolist(),
            labels[2]: series3_c_v.tolist() + series3_t_v.tolist(),
            labels[3]: series4_c_v.tolist() + series4_t_v.tolist(),
        },
        title= condition + " " + marker + " Nostril Comparison in " + area,
        ylabel="Cells / mm^2",
        broken=False,
        bool_offset=0.15,
        point_colors={
            labels[0]: [False] * len(series1_c_v.tolist()) + [True] * len(series1_t_v.tolist()),
            labels[1]: [False] * len(series2_c_v.tolist()) + [True] * len(series2_t_v.tolist()),
            labels[2]: [False] * len(series3_c_v.tolist()) + [True] * len(series3_t_v.tolist()),
            labels[3]: [False] * len(series4_c_v.tolist()) + [True] * len(series4_t_v.tolist())
        },
        color_false='paleturquoise',
        color_true='gold',
        color_legend=("Non-Injected", "Injected"),
        figsize=(8, 3.5),
        save_path="Figures/" + condition + "_Sides_" + marker
    )

def plot_d_vs_v(df, marker, condition, labels, area):
    area = "OE" if area is None else area

    control_df = df[df['Injection'] == "C"]
    treatment_df = df[df['Injection'] == "T"]

    if condition == 'LPS':
        #Section Names using LPS convention (TBXMN)
        dorsal_df = df[(df['Section'] == "N") | (df['Section'] == "M")]
        ventral_df = df[(df['Section'] == "B") | (df['Section'] == "T") | (df['Section'] == "X")]

        timepoint1 = df[(df['Time'] == "6h") & (df['Condition'] == "Saline")]
        timepoint2 = df[(df['Time'] == "6h") & (df['Condition'] == condition)]
        timepoint3 = df[(df['Time'] == "2d") & (df['Condition'] == condition)]
        timepoint4 = df[(df['Time'] == "5d") & (df['Condition'] == condition)]

    elif condition == 'PolyIC':
        #Section Names using Proper Convention (DR, DM, VM, VLA, VLB)
        dorsal_df = df[(df['Section'] == "DR") | (df['Section'] == "DM")]
        ventral_df = df[(df['Section'] == "VLA") | (df['Section'] == "VLB") | (df['Section'] == "VM")]

        timepoint1 = df[(df['Time'] == "6h") & (df['Condition'] == condition)]
        timepoint2 = df[(df['Time'] == "2d") & (df['Condition'] == condition)]
        timepoint3 = df[(df['Time'] == "5d") & (df['Condition'] == condition)]
        timepoint4 = df[(df['Time'] == "8d") & (df['Condition'] == condition)]

    series1_t_v = intersect_dfs([treatment_df, ventral_df, timepoint1])[area]
    series2_t_v = intersect_dfs([treatment_df, ventral_df, timepoint2])[area]
    series3_t_v = intersect_dfs([treatment_df, ventral_df, timepoint3])[area]
    series4_t_v = intersect_dfs([treatment_df, ventral_df, timepoint4])[area]

    series1_t_d = intersect_dfs([treatment_df, dorsal_df, timepoint1])[area]
    series2_t_d = intersect_dfs([treatment_df, dorsal_df, timepoint2])[area]
    series3_t_d = intersect_dfs([treatment_df, dorsal_df, timepoint3])[area]
    series4_t_d = intersect_dfs([treatment_df, dorsal_df, timepoint4])[area]

    generate_plot(
        groups={
            labels[0]: series1_t_v.tolist() + series1_t_d.tolist(),
            labels[1]: series2_t_v.tolist() + series2_t_d.tolist(),
            labels[2]: series3_t_v.tolist() + series3_t_d.tolist(),
            labels[3]: series4_t_v.tolist() + series4_t_d.tolist(),
        },
        title= condition + " " + marker + " in Dorsal vs Ventral OE injected",
        ylabel="Cells / mm^2",
        broken=False,
        bool_offset=0.15,
        point_colors={
            labels[0]: [False] * len(series1_t_v.tolist()) + [True] * len(series1_t_d.tolist()),
            labels[1]: [False] * len(series2_t_v.tolist()) + [True] * len(series2_t_d.tolist()),
            labels[2]: [False] * len(series3_t_v.tolist()) + [True] * len(series3_t_d.tolist()),
            labels[3]: [False] * len(series4_t_v.tolist()) + [True] * len(series4_t_d.tolist())
        },
        color_false='seagreen',
        color_true='lightcoral',
        color_legend=("Ventral", "Dorsal"),
        figsize=(8, 3.5),
        save_path="Figures/" + condition + "_DV_" + marker
    )

def plot_for_marker(marker_name, condition):
    if condition == "LPS":
        marker_df = LPS_marker_dfs[marker_name]
        labels = [
            "Saline 6h",
            "LPS 6h",
            "LPS 2d",
            "LPS 5d"
        ]
    elif condition == "PolyIC":
        marker_df = PIC_marker_dfs[marker_name]
        labels = [
            "PolyIC 6h",
            "PolyIC 2d",
            "PolyIC 5d",
            "PolyIC 8d"
        ]
    plot_oe_v_lp(marker_df, marker_name, condition, labels)
    plot_c_vs_t(marker_df, marker_name, condition, labels, "OE")
    plot_d_vs_v(marker_df, marker_name, condition, labels, "OE")

# plot_for_marker("S100A9", "LPS")
# plot_for_marker("CD3", "LPS")
# plot_for_marker("Ly6G", "LPS")
# plot_for_marker("CC3", "LPS")
# plot_for_marker("CD11B", "LPS")

# plot_for_marker("S100A9", "PolyIC")
# plot_for_marker("CD3", "PolyIC")
# plot_for_marker("CD45", "PolyIC")
# plot_for_marker("EGR1", "PolyIC")

# #Special cases
# plot_c_vs_t(LPS_marker_dfs[marker_name], "CD3", "LPS", "OE")
# plot_c_vs_t(LPS_marker_dfs[marker_name], "CD3", "LPS", "LP")